# E.R.M.E.S. - Fase 5: Explainable AI (Grad-CAM) e Fondamenti Matematici

L'Accuratezza statistica non è sufficiente per validare un modello destinato all'interazione uomo-macchina. In ottemperanza ai principi dell'**AI Responsabile** e dell'**Interpretabilità**, in questo notebook "apriamo" la scatola nera della nostra Rete Neurale Convoluzionale utilizzando l'algoritmo **Grad-CAM (Gradient-weighted Class Activation Mapping)**.

### Formalizzazione Matematica dell'Algoritmo
Il nucleo del Grad-CAM si articola in due fasi matematiche che non richiedono alterazioni architetturali o ri-addestramenti, ma sfruttano la differenziazione automatica dei tensori:

1. **Calcolo dei coefficienti di importanza:** Viene calcolato il gradiente del punteggio logit della classe predetta $c$ (indicato con $y^c$) rispetto alle singole attivazioni $A^k$ dell'ultimo layer convoluzionale. Applicando un *Global Average Pooling* su tali gradienti, si ottiene il coefficiente $\alpha_k^c$ per l'intera *feature map*:
$$\alpha_k^c = \frac{1}{Z} \sum_{i} \sum_{j} \frac{\partial y^c}{\partial A^k_{i,j}}$$
Dove $Z$ rappresenta la dimensionalità spaziale della mappa (larghezza moltiplicata per altezza). Questo coefficiente quantifica matematicamente l'impatto della singola mappa di attivazione sulla predizione finale.

2. **Generazione della Mappa di Calore:** La mappa finale $L^c_{\text{Grad-CAM}}$ viene generata operando una combinazione lineare pesata delle *feature maps*. Un passaggio cruciale è l'applicazione della funzione di attivazione **ReLU**:
$$L^c_{\text{Grad-CAM}} = \text{ReLU} \left( \sum_k \alpha_k^c A^k \right)$$
L'uso della ReLU garantisce che vengano isolate esclusivamente le caratteristiche con influenza *positiva* sulla classe analizzata, azzerando i gradienti negativi associati a pattern inibitori (ovvero i pixel che spingono la rete a classificare l'immagine in un'altra categoria).

In [ ]:
"""
Configurazione dell'ambiente per l'Explainable AI.
Importiamo il modello custom salvato durante l'addestramento e 
identifichiamo l'ultimo layer convoluzionale utile per l'estrazione dei gradienti.
"""
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from pathlib import Path
import random
import logging

tf.get_logger().setLevel(logging.ERROR)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context("notebook", font_scale=1.1)

print("--- INIZIALIZZAZIONE EXPLAINABLE AI ---")

# 1. Caricamento del Modello Addestrato
try:
    model = tf.keras.models.load_model('ermes_best_cnn.h5')
    print("Modello caricato con successo dalla cartella corrente")
except:
    model = tf.keras.models.load_model('../models/ermes_best_cnn.h5')
    print("Modello caricato con successo dalla cartella '../models/'")

# 2. Identificazione dinamica dell'ultimo layer convoluzionale
last_conv_layer_name = None
for layer in reversed(model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer_name = layer.name
        break

print(f"Ultimo layer convoluzionale identificato: '{last_conv_layer_name}'")

# Label delle emozioni in ordine alfabetico
emotions = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

In [ ]:
def make_gradcam_heatmap(img_array: tf.Tensor, model: tf.keras.Model, last_conv_layer_name: str, pred_index=None) -> np.ndarray:
    """
    Algoritmo nucleo del Grad-CAM.
    Mappa le operazioni tensoriali direttamente sulle formule matematiche teoriche.
    """
    grad_model = tf.keras.models.Model(
        [model.inputs], 
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    # GradientTape registra le operazioni per la differenziazione automatica (calcolo derivate)
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        
        # Estrazione del logit della classe predetta (y^c nella formula)
        class_channel = preds[:, pred_index]

    # Derivata parziale della classe predetta rispetto alle feature map: (∂y^c / ∂A^k)
    grads = tape.gradient(class_channel, last_conv_layer_output)

    # Formula 1: Calcolo di α_k^c tramite Global Average Pooling vettoriale dei gradienti
    # tf.reduce_mean esegue la sommatoria divisa per Z sull'asse spaziale (0, 1, 2)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Formula 2 (Parte A): Combinazione lineare pesata (∑ α_k^c * A^k)
    # Moltiplichiamo ogni canale della feature map per il suo coefficiente di importanza
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Formula 2 (Parte B): Applicazione della funzione ReLU
    # tf.maximum(heatmap, 0) scarta i gradienti negativi (pattern inibitori)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    
    return heatmap.numpy()

In [ ]:
def display_gradcam(img_path: Path, heatmap: np.ndarray, title: str, ax: plt.Axes, color: str = 'black') -> None:
    """
    Funzione di rendering grafico.
    Ridimensiona la mappa di calore, applica una scala cromatica (JET) 
    e la sovrappone in trasparenza all'immagine originale in scala di grigi.
    """
    # Lettura immagine originale grezza
    img = cv2.imread(str(img_path))
    if img is None:
        raise ValueError(f"Impossibile leggere l'immagine al percorso: {img_path}")
    
    # Resize della heatmap per farla combaciare con l'immagine originale
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    
    # Conversione in scala cromatica (Rosso = Max attivazione, Blu = Minima attivazione)
    heatmap = np.uint8(255 * heatmap)
    heatmap_colored = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    
    # Sovrapposizione bilanciata: 40% Heatmap + 60% Immagine Originale
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    superimposed_img = heatmap_colored * 0.4 + img_rgb * 0.6
    superimposed_img = np.clip(superimposed_img, 0, 255).astype(np.uint8)
    
    # Disegno sul canvas Matplotlib
    ax.imshow(superimposed_img)
    ax.set_title(title, fontsize=13, fontweight='bold', color=color, pad=10)
    ax.axis('off')

In [ ]:
"""
Esecuzione del test visivo su un campione casuale e 
stampa dei coefficienti matematici alfa (α_k^c) del Grad-CAM.
"""
test_dir = Path('../data/fer2013/test')

fig, axes = plt.subplots(2, len(emotions), figsize=(22, 7))
fig.suptitle('Explainable AI: Analisi Grad-CAM e Pesi Matematici (Modello E.R.M.E.S.)', fontsize=18, fontweight='bold', y=1.02)

print("--- ESTRAZIONE GRADIENTI E COEFFICIENTI (α_k^c) ---")

for i, emotion in enumerate(emotions):
    emotion_path = test_dir / emotion
    all_images = list(emotion_path.glob('*'))
    img_path = random.choice(all_images)
    
    # Preparazione immagine
    img_raw = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    img_tensor = img_raw / 255.0 
    img_tensor = np.expand_dims(img_tensor, axis=-1) 
    img_tensor = np.expand_dims(img_tensor, axis=0)  
    
    # Predizione
    preds = model.predict(img_tensor, verbose=0)
    pred_idx = np.argmax(preds[0])
    pred_emotion = emotions[pred_idx]
    confidence = preds[0][pred_idx] * 100
    
    # Generazione Grad-CAM e recupero modello per estrarre i pesi alfa
    heatmap = make_gradcam_heatmap(tf.convert_to_tensor(img_tensor), model, last_conv_layer_name)
    
    # --- CALCOLO MATEMATICO PER IL REPORT ---
    grad_model = tf.keras.models.Model([model.inputs], [model.get_layer(last_conv_layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_tensor)
        class_channel = predictions[:, pred_idx]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2)).numpy() # Questi sono gli alfa
    
    # Troviamo i 3 filtri convoluzionali più importanti (con l'alfa maggiore)
    top_3_filters = np.argsort(pooled_grads)[-3:][::-1]
    
    print(f"[{emotion.upper()}] -> Predetto: {pred_emotion} ({confidence:.1f}%)")
    print(f"    Top 3 Filtri (Coefficienti α):")
    for f_idx in top_3_filters:
        print(f"    - Filtro n.{f_idx}: α = {pooled_grads[f_idx]:.5f}")
    print("-" * 50)
    
    # Rendering
    axes[0, i].imshow(img_raw, cmap='gray')
    axes[0, i].set_title(f"Reale: {emotion.upper()}", fontsize=14, fontweight='bold', pad=10)
    axes[0, i].axis('off')
    
    title_color = '#2ca02c' if emotion == pred_emotion else '#d62728'
    title_gradcam = f"Predetto: {pred_emotion}\nConfidenza: {confidence:.1f}%"
    display_gradcam(img_path, heatmap, title_gradcam, axes[1, i], color=title_color)

plt.tight_layout()
plt.savefig('gradcam_analysis.svg', format='svg', bbox_inches='tight')
print("\nAnalisi completata.")
plt.show()

## Conclusioni della Fase 5

L'estrazione esplicita dei coefficienti di importanza spaziale ($\alpha_k^c$) tramite Grad-CAM fornisce una radiografia matematica del processo decisionale della rete E.R.M.E.S. V1. L'analisi incrociata tra i log convoluzionali e l'ispezione visiva delle mappe di calore permette di formalizzare tre deduzioni ingegneristiche fondamentali:

1. **Routing delle Feature e Incertezza Predittiva:** I log dimostrano che la rete instrada l'input verso percorsi neurali specifici in funzione dell'astrazione visiva. Il campione *Fear*, misclassificato come *Neutral*, attiva esattamente la medesima triade di filtri convoluzionali (n.162, n.109, n.111) del campione *Neutral* reale. Questo certifica che l'errore non è stocastico, ma deriva da un instradamento vettoriale ambiguo all'interno dell'iperspazio delle feature, aggravato da una scarsa confidenza predittiva (31.2%).
2. **Ancoraggio Spaziale e Robustezza al Rumore:** Le mappe di calore evidenziano un focus su specifici distretti muscolari. Nei campioni *Angry*, *Disgust* e *Sad* (misclassificati in *Angry*), le aree di massima attivazione collassano sistematicamente sulla glabella. La rete ha appreso ad associare il corrugamento frontale alla rabbia (confidenza 95.2% su *Disgust*). Tuttavia, la rete si dimostra strutturalmente immune agli artefatti: nel medesimo campione *Disgust*, nonostante un vistoso watermark sovrimpresso in basso, l'attenzione si mantiene ancorata alla morfologia facciale, confermando l'efficacia del MaxPooling.
3. **Apprendimento di Bias Contestuali:** L'approccio XAI fa emergere come la rete compensi la bassa risoluzione assimilando pattern secondari. Nel campione *Surprise*, la mappa di calore non si limita ai tratti somatici, ma si estende vividamente sulle mani appoggiate alle guance e su elementi di disturbo periferici (fermaglio nei capelli). La rete ha imparato a utilizzare il gesto delle mani al volto come "scorciatoia predittiva" ricorrente nel dataset.

**Sviluppi Futuri (Verso la Fase 6):** Queste evidenze certificano che l'architettura E.R.M.E.S. V1 modella relazioni geometricamente coerenti, ma ha saturato la propria capacità espressiva (limitata a ~5.87 milioni di parametri). Per estendere l'asintoto prestazionale e approssimare l'Upper Bound teorico (~65%), l'unico step ingegneristico risolutivo è il **Transfer Learning**. 
Nella Fase 6, abbandoneremo l'architettura *custom* per importare la **VGG16**, un modello pre-addestrato sul vasto dominio visivo di ImageNet. Valuteremo empiricamente se una *Model Capacity* enormemente superiore (~27.7 milioni di parametri addestrabili), unita all'Upsampling tensoriale (da 48x48 a 224x224 pixel), sia in grado di superare i limiti fisiologici di informazione del dataset FER-2013.